In [1]:
import pandas as pd
import numpy as np
import re

In [6]:
data = pd.read_csv(r"C:/Users/QJ095K/Desktop/2026-03-10 Liste Faltschachteln.csv", sep=';',header=0,encoding='ISO-8859-1')
df = pd.DataFrame(data=data)
#Leere Spalten dropen
df_clean=df.dropna(axis=1,how='all')
df_clean.drop(columns=["Spalte6","Spalte7"])
#Materialnummer aus Object ID extrahieren, um SAP Eingabe zu ermöglichen
df_clean["materialNumber"] = (
    df_clean["Object ID"]
    .astype(str)
    .str.extract(r"^0*([0-9]+)(?:/00)?$")[0]
)
#nur die wichtigen Spalten beibehalten
df_min = df_clean[['Object Name',"materialNumber","Object ID","Length in mm internal","Width in mm internal","Height in mm internal","Amount per bin box (LHM C)","Status"]]
#Maße aus dem ObjectName extrahieren, da hier die innenmaße angegeben sind
pattern = r'(\d+)\s*[Xx*]\s*(\d+)\s*[Xx*]\s*(\d+)'
df_min[['length', 'width', 'height']] = df_min['Object Name'].str.extract(pattern)

df_min['Stabilität'] = (
    df_min['Object Name']
    .str.extract(r'\b(1E|1B)\b', expand=False)
    .fillna('keine')
)

In [7]:
print(df_min.count())
df_min = df_min[df_min['Status'] != '99 - Inactive']
print(df_min.count())

Object Name                   1882
materialNumber                1857
Object ID                     1884
Length in mm internal         1553
Width in mm internal          1554
Height in mm internal         1539
Amount per bin box (LHM C)    1032
Status                        1822
length                        1586
width                         1586
height                        1586
Stabilität                    1884
dtype: int64
Object Name                   1402
materialNumber                1380
Object ID                     1404
Length in mm internal         1278
Width in mm internal          1279
Height in mm internal         1268
Amount per bin box (LHM C)     962
Status                        1342
length                        1239
width                         1239
height                        1239
Stabilität                    1404
dtype: int64


In [8]:
#Für die Verpackungen, bei denen die Maße nicht extrahiert werden konnten, die Standardmaße aus der Tabelle verwenden
mask = df_min['length'].isna()

source_cols = ["Length in mm internal",
               "Width in mm internal",
               "Height in mm internal"]

target_cols = ['length', 'width', 'height']

df_min.loc[mask, target_cols] = df_min.loc[mask, source_cols].values
df_min[df_min['length'].isna()].shape[0]
df_clean = df_min.dropna(subset=['height'])

spalten = ['length', 'height', 'width']

#Punkte entfernen und Werte in Text umwandeln
for col in spalten:
    df_clean[col] = df_clean[col].astype(str).str.replace('.', '', regex=False)
#Werte in Zahlen umwandeln
df_clean['length']= pd.to_numeric(df_clean['length'])
df_clean['height']= pd.to_numeric(df_clean['height'])
df_clean['width']= pd.to_numeric(df_clean['width'])
#Ausschließen von Verpackungsmaterial die nur 2-dimensional sind (Zwischenpappen,...)
df_clean=df_clean[df_clean['height']!=0]
#Nur die relevanten Spalten
Verpackungen= df_clean[['Object Name','length','height','width','materialNumber',"Amount per bin box (LHM C)","Stabilität"]]
#materialNumber als Index festlegen und in string umwandeln
Verpackungen.set_index("materialNumber")
Verpackungen["materialNumber"] = Verpackungen["materialNumber"].astype(str)

### Dokument erstellen

In [9]:
Verpackungen.to_csv('meine_datei.csv',sep=',', index=False, encoding='utf-8')

## Mengen der Verpackungen einfügen

In [6]:
sapData = pd.read_csv("C:/Users/QJ095K/Documents/SAP/SAP GUI/.txt",sep=";", encoding="utf-8")
#sapData =sapData.drop(columns=["MengeKB","EinheitKB"])
sapData["Material"]= sapData["Material"].astype("str")
sapData

,Material,WertBB,EinheitBBMengeBB,Einheit,MengeBB,MengeKB,EinheitKB
0,10437,"0,00",EUR,0.0,ST,0,ST
1,32525,"0,00",EUR,0.0,ST,0,ST
2,42991,"0,00",***,0.0,ST,0,ST
3,44478,"0,00",EUR,0.0,ST,0,ST
4,46003,"0,00",EUR,0.0,ST,0,ST
...,...,...,...,...,...,...,...
798,52005126,"0,00",EUR,0.0,ST,0,ST
799,52006571,"0,00",EUR,0.0,ST,0,ST
800,52006572,"0,00",EUR,0.0,ST,0,ST
801,52006771,"2.268,90",EUR,172.0,ST,0,ST


In [7]:
Verpackungen_merged = Verpackungen.merge(
    sapData,
    left_on="materialNumber",
    right_on="Material",
    how="left"
)
Verpackungen_merged

,Object Name,length,height,width,materialNumber,Amount per bin box (LHM C),Material,WertBB,EinheitBBMengeBB,Einheit,MengeBB,MengeKB,EinheitKB
0,PAPERBOX 0421 GD2 32X15X95,32,95,15,1041201,NaN,1041201,"7.077,96",PLN,7.934,ST,0.0,ST
1,PAPERBOX A2320 GZ1 HÜLLE DS,35000,75000,10000,153913,1033,153913,"65,87",EUR,8.000,ST,0.0,ST
2,PAPERBOX 1E 0215 106X41X78 BL,106,78,41,140125,77,140125,"0,00",EUR,0.000,ST,0.0,ST
3,PAPERBOX 1E 0215 106X41X78 P,106,78,41,9051692,77,9051692,"140,87",EUR,1.286,ST,0.0,ST
4,PAPERBOX 1E 0426 103X50X25 P,103,25,50,9501429,180,9501429,"1.419,44",***,10.288,ST,0.0,ST
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1338,GENERAL CARTON 635X260X270,635,270,260,9620054,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1339,RAIL CARTON 1015X190X60,1015,60,190,9620096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1340,FALTSCH. 307X 67X 47,307,47,67,9621859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1341,UMZUGKARTON 650X350X370 2.4,650,370,350,9647325,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
#Verpackungen_merged.sort_values("Wert BB", ascending=False)
print(Verpackungen_merged[Verpackungen_merged["WertBB"].isna()])


                            Object Name  length  height  width materialNumber  \
7                      EPE BOX 60X25X35      60      35     25         194224   
8          PAPERBOX 0426 1E 66X82X95 BL      66      95     82        1018133   
9          PAPERBOX 0713 1E 66X66X115 S      66     115     66        1026011   
10        PAPERBOX 0713 1E 66X66X115 BL      66     115     66        1026168   
14    PAPERBOX 70X55X72 FOR FUSEADAPTER      70      72     55        1296482   
...                                 ...     ...     ...    ...            ...   
1337              PAPER BOX 800X255X250     800     250    255        9620025   
1338         GENERAL CARTON 635X260X270     635     270    260        9620054   
1339            RAIL CARTON 1015X190X60    1015      60    190        9620096   
1340               FALTSCH. 307X 67X 47     307      47     67        9621859   
1341        UMZUGKARTON 650X350X370 2.4     650     370    350        9647325   

     Amount per bin box (LH

In [9]:
print(Verpackungen_merged[Verpackungen_merged["WertBB"].notna()])

                        Object Name  length  height  width materialNumber  \
0        PAPERBOX 0421 GD2 32X15X95      32      95     15        1041201   
1       PAPERBOX A2320 GZ1 HÜLLE DS   35000   75000  10000         153913   
2     PAPERBOX 1E 0215 106X41X78 BL     106      78     41         140125   
3      PAPERBOX 1E 0215 106X41X78 P     106      78     41        9051692   
4      PAPERBOX 1E 0426 103X50X25 P     103      25     50        9501429   
...                             ...     ...     ...    ...            ...   
1330  PAPERBOX 1E 0426 316X89X50 BL     316      50     89        9796755   
1331  PAPERBOX 1E 0426 238X68X49 BL     238      49     68        9796768   
1332  PAPERBOX 1E 0426 222X91X53 BL     222      53     91        9796771   
1334   WELLPAPPK.392X301X5 KALENDER     392       5    301        9503155   
1342       TVP SE S02 OT 400X290X60     400      60    290        9799927   

     Amount per bin box (LHM C) Material    WertBB EinheitBBMengeBB  Einhei

In [10]:
Verpackungen_merged.to_csv('meine_datei.csv',sep=',', index=False, encoding='utf-8')